In [1]:
pip install settrade-v2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import psycopg2

# เชื่อมต่อฐานข้อมูล
conn = psycopg2.connect(
    host="localhost",
    database="thai_stocks_db",
    user="postgres",
    password="Shifa.326459"
)
cur = conn.cursor()

# ลบตารางเก่าทิ้งก่อน (ถ้าอยากเริ่มใหม่สะอาดๆ)
cur.execute("DROP TABLE IF EXISTS stocks CASCADE;")
cur.execute("DROP TABLE IF EXISTS sectors CASCADE;")

# สร้างตาราง Sectors
cur.execute("""
CREATE TABLE sectors (
    sector_id SERIAL PRIMARY KEY,
    sector_name VARCHAR(50) UNIQUE NOT NULL
);
""")

# สร้างตาราง Stocks (มี Foreign Key เชื่อมกับ sectors)
cur.execute("""
CREATE TABLE stocks (
    symbol VARCHAR(20) PRIMARY KEY,
    description_en TEXT,
    description_th TEXT,
    sector_id INTEGER REFERENCES sectors(sector_id)
);
""")

conn.commit()
print("--- สร้าง Schema สำหรับรายชื่อหุ้นเสร็จสมบูรณ์ ---")

--- สร้าง Schema สำหรับรายชื่อหุ้นเสร็จสมบูรณ์ ---


In [15]:
import psycopg2

# 1. ตั้งค่าการเชื่อมต่อ
DB_CONFIG = {
    "host": "127.0.0.1",
    "database": "thai_stocks_db",
    "user": "postgres",
    "password": "Shifa.326459"
}

# 2. เชื่อมต่อ (ระวังวงเล็บนะครับ!)
try:
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    cur = conn.cursor()
    print("[SUCCESS] Database Connection Established.")
    
    # 3. ลองสร้างตารางแรกเพื่อทดสอบ
    cur.execute("CREATE TABLE IF NOT EXISTS sectors (sector_id SERIAL PRIMARY KEY, sector_name VARCHAR(100) UNIQUE);")
    print("[SUCCESS] Table 'sectors' is ready.")

except Exception as e:
    print(f"[ERROR] Connection failed: {e}")

[SUCCESS] Database Connection Established.
[SUCCESS] Table 'sectors' is ready.


In [17]:
import psycopg2

# เชื่อมต่อ Database (ใช้ข้อมูลของคุณ)
conn = psycopg2.connect(
    host="127.0.0.1",
    database="thai_stocks_db",
    user="postgres",
    password="Shifa.326459" # 
)
conn.autocommit = True
cur = conn.cursor()

# สร้างตาราง stocks 
cur.execute("""
CREATE TABLE IF NOT EXISTS stocks (
    symbol VARCHAR(20) PRIMARY KEY,
    instrument_type VARCHAR(20),
    security_type VARCHAR(20),
    market_status VARCHAR(20),
    pe NUMERIC(10, 2),
    pbv NUMERIC(10, 2),
    eps NUMERIC(10, 2),
    div_yield NUMERIC(10, 2)
);
""")

print("[SUCCESS] Table 'stocks' created successfully.")

[SUCCESS] Table 'stocks' created successfully.


In [22]:
import psycopg2
from settrade_v2 import Investor

try:
    # 1. เชื่อมต่อ Database
    conn = psycopg2.connect(
        host="127.0.0.1",
        database="thai_stocks_db",
        user="postgres",
        password="Shifa.326459"
    )
    conn.autocommit = True
    cur = conn.cursor()
    print("1. Database Connected ✅")

    # 2. เชื่อมต่อ API
    investor = Investor(
        app_id="m6P7C2ceQ5ffMv7k", 
        app_secret="AIlXY/4yHMpjZN5WthcCGDSGSyL8XWsh0WdyAnFWbzxo", 
        broker_id="SANDBOX",
        app_code="SANDBOX"
    )
    market = investor.MarketData()
    
    # ดึงข้อมูลมาเช็คหน้าตา
    raw_res = market.get_symbols()
    print(f"2. API Response type: {type(raw_res)}")
    
    # ดึง List หุ้นออกมา (ปกติจะเป็น dict ที่มี key ว่า 'symbols')
    if isinstance(raw_res, dict) and 'symbols' in raw_res:
        all_symbols = raw_res['symbols']
    else:
        all_symbols = raw_res # เผื่อบางเวอร์ชันคืนเป็น list เลย
    
    print(f"3. Found symbols in API: {len(all_symbols)}")

    # 3. บันทึกลง Database
    for symbol in all_symbols:
        # ใช้คำสั่ง INSERT แบบง่ายที่สุด
        cur.execute("INSERT INTO stocks (symbol) VALUES (%s) ON CONFLICT (symbol) DO NOTHING;", (symbol,))

    # 4. นับจำนวนจริงใน DB
    cur.execute("SELECT COUNT(*) FROM stocks;")
    total_stocks = cur.fetchone()[0]

    print(f"\n====================================")
    print(f"✅ HOW MANY STOCKS? : {total_stocks} ตัว")
    print(f"====================================")

except Exception as e:
    print(f" พบข้อผิดพลาด: {e}")

1. Database Connected ✅
 พบข้อผิดพลาด: 'MarketData' object has no attribute 'get_symbols'


In [27]:
import psycopg2
from settrade_v2 import Investor
import time

# 1. เชื่อมต่อ Database
conn = psycopg2.connect(
    host="127.0.0.1",
    database="thai_stocks_db",
    user="postgres",
    password="Shifa.326459" # รหัสของคุณ
)
conn.autocommit = True
cur = conn.cursor()

print("🔨 กำลังปรับปรุงโครงสร้าง Database...")

# --- STEP A: ล้างตารางเก่าทิ้ง (Drop) และสร้างใหม่ (Create) ---
# ใช้ CASCADE เพื่อลบความสัมพันธ์ที่อาจค้างอยู่
cur.execute("DROP TABLE IF EXISTS daily_prices CASCADE;") 
cur.execute("DROP TABLE IF EXISTS stocks CASCADE;")
cur.execute("DROP TABLE IF EXISTS sectors CASCADE;")

# 1. สร้างตาราง Sectors ใหม่
cur.execute("""
CREATE TABLE sectors (
    sector_id SERIAL PRIMARY KEY,
    sector_name VARCHAR(100) UNIQUE
);
""")

# 2. สร้างตาราง Stocks ใหม่ (ให้มี instrument_type ตามที่โค้ดต้องการ)
cur.execute("""
CREATE TABLE stocks (
    symbol VARCHAR(20) PRIMARY KEY,
    instrument_type VARCHAR(20),  -- เพิ่มตัวนี้เข้ามาแก้ Error
    market_status VARCHAR(20),
    pe NUMERIC,
    div_yield NUMERIC,
    sector_id INTEGER REFERENCES sectors(sector_id)
);
""")

print("✅ สร้างตารางใหม่เสร็จเรียบร้อย! (มี instrument_type แล้ว)")

# --- STEP B: เชื่อมต่อ API ---
investor = Investor(
    app_id="m6P7C2ceQ5ffMv7k", 
    app_secret="AIlXY/4yHMpjZN5WthcCGDSGSyL8XWsh0WdyAnFWbzxo", 
    broker_id="SANDBOX",
    app_code="SANDBOX"
)
market = investor.MarketData()

# --- STEP C: เตรียมข้อมูล (Seeding) ---
# รายชื่อหุ้นตัวอย่าง (Top Stocks)
target_stocks = [
    "ADVANC", "AOT", "BBL", "BDMS", "BEM", "BGRIM", "BH", "BTS", "CBG", "CPALL",
    "CPF", "CPN", "CRC", "DELTA", "EA", "EGCO", "GLOBAL", "GPSC", "GULF", "HMPRO",
    "INTUCH", "IVL", "KBANK", "KTB", "KTC", "LH", "MINT", "MTC", "OR", "OSP",
    "PTT", "PTTEP", "PTTGC", "RATCH", "SAWAD", "SCB", "SCC", "SCGP", "TISCO", "TOP",
    "TRUE", "TTB", "TU", "WHA"
]

print(f"📦 กำลังบันทึกข้อมูลหุ้น {len(target_stocks)} ตัว...")

count = 0
for symbol in target_stocks:
    try:
        # ดึงข้อมูลจาก Sandbox
        info = market.get_quote_symbol(symbol)
        
        # เตรียมข้อมูล (Handle missing data)
        sec_name = info.get('sector', 'SET100')
        pe = info.get('pe', 0)
        yield_val = info.get('percentYield', 0)
        mkt_status = info.get('marketStatus', 'OPEN')
        
        # 1. จัดการ Sector
        cur.execute("INSERT INTO sectors (sector_name) VALUES (%s) ON CONFLICT (sector_name) DO NOTHING", (sec_name,))
        cur.execute("SELECT sector_id FROM sectors WHERE sector_name = %s", (sec_name,))
        s_id = cur.fetchone()[0]

        # 2. จัดการ Stock
        cur.execute("""
            INSERT INTO stocks (symbol, instrument_type, market_status, pe, div_yield, sector_id)
            VALUES (%s, 'STOCK', %s, %s, %s, %s)
            ON CONFLICT (symbol) DO NOTHING;
        """, (symbol, mkt_status, pe, yield_val, s_id))
        
        print(f"   -> Saved: {symbol}")
        count += 1
        time.sleep(0.05) # กันเหนียว

    except Exception as e:
        print(f"⚠️ Error {symbol}: {e}")

# --- STEP D: สรุปผล ---
cur.execute("SELECT COUNT(*) FROM stocks;")
total = cur.fetchone()[0]

print(f"\n====================================")
print(f"🎉 สำเร็จ! แก้ไข Schema และบันทึกข้อมูลแล้ว")
print(f"📊 จำนวนหุ้นใน Database: {total} ตัว")
print(f"====================================")

🔨 กำลังปรับปรุงโครงสร้าง Database...
✅ สร้างตารางใหม่เสร็จเรียบร้อย! (มี instrument_type แล้ว)
📦 กำลังบันทึกข้อมูลหุ้น 44 ตัว...
   -> Saved: ADVANC
   -> Saved: AOT
   -> Saved: BBL
   -> Saved: BDMS
   -> Saved: BEM
   -> Saved: BGRIM
   -> Saved: BH
   -> Saved: BTS
   -> Saved: CBG
   -> Saved: CPALL
   -> Saved: CPF
   -> Saved: CPN
   -> Saved: CRC
   -> Saved: DELTA
   -> Saved: EA
   -> Saved: EGCO
   -> Saved: GLOBAL
   -> Saved: GPSC
   -> Saved: GULF
   -> Saved: HMPRO
⚠️ Error INTUCH: Symbol not found
   -> Saved: IVL
   -> Saved: KBANK
   -> Saved: KTB
   -> Saved: KTC
   -> Saved: LH
   -> Saved: MINT
   -> Saved: MTC
   -> Saved: OR
   -> Saved: OSP
   -> Saved: PTT
   -> Saved: PTTEP
   -> Saved: PTTGC
   -> Saved: RATCH
   -> Saved: SAWAD
   -> Saved: SCB
   -> Saved: SCC
   -> Saved: SCGP
   -> Saved: TISCO
   -> Saved: TOP
   -> Saved: TRUE
   -> Saved: TTB
   -> Saved: TU
   -> Saved: WHA

🎉 สำเร็จ! แก้ไข Schema และบันทึกข้อมูลแล้ว
📊 จำนวนหุ้นใน Database: 43 ตัว


In [28]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. ตอบคำถาม: มีกี่กลุ่ม? และมีกลุ่มอะไรบ้าง?
sql_sectors = "SELECT * FROM sectors ORDER BY sector_name;"
df_sectors = pd.read_sql(sql_sectors, conn)

print(f"✅ Q: How many stock categories?")
print(f"👉 A: {len(df_sectors)} Categories")
print(f"\n✅ Q: List of categories?")
print(df_sectors)

# ---------------------------------------------------------

# 2. ตอบคำถาม: แต่ละกลุ่มมีหุ้นกี่ตัว? (Stock count per category)
# เราต้องใช้ SQL JOIN เพื่อเชื่อมตาราง stocks กับ sectors เข้าด้วยกัน
sql_count = """
SELECT s.sector_name, COUNT(st.symbol) as total_stocks
FROM sectors s
JOIN stocks st ON s.sector_id = st.sector_id
GROUP BY s.sector_name
ORDER BY total_stocks DESC;
"""
df_count = pd.read_sql(sql_count, conn)

print(f"\n✅ Q: Summary of stocks in each category")
print(df_count)


✅ Q: How many stock categories?
👉 A: 1 Categories

✅ Q: List of categories?
   sector_id sector_name
0          1      SET100

✅ Q: Summary of stocks in each category
  sector_name  total_stocks
0      SET100            43


C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_7528\2022608929.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sectors = pd.read_sql(sql_sectors, conn)
C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_7528\2022608929.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_count = pd.read_sql(sql_count, conn)


In [29]:

# 2. สร้าง Mapping ข้อมูลจริง (Manual Mapping)
sector_mapping = {
    'ENERG': ['PTT', 'PTTEP', 'TOP', 'GPSC', 'GULF', 'EA', 'BGRIM', 'EGCO', 'RATCH', 'OR', 'PTTGC', 'BANPU'],
    'BANK': ['KBANK', 'SCB', 'BBL', 'KTB', 'TTB', 'TISCO'],
    'COMM': ['CPALL', 'CRC', 'HMPRO', 'GLOBAL'],
    'ICT': ['ADVANC', 'TRUE', 'INTUCH'],
    'TRANS': ['AOT', 'BTS', 'BEM'],
    'HELTH': ['BDMS', 'BH'],
    'ETRON': ['DELTA'], # Electronic
    'FOOD': ['CPF', 'TU', 'MINT', 'OSP', 'CBG'],
    'PROP': ['CPN', 'LH', 'WHA', 'AWC'], # Property
    'FIN': ['MTC', 'SAWAD', 'KTC'] # Finance
}

print("🔄 กำลังอัปเดต Sector ให้ถูกต้องตามความจริง...")

# 3. วนลูปอัปเดตข้อมูลลง Database
for sector_name, symbols in sector_mapping.items():
    # 3.1 สร้าง Sector ใหม่ถ้ายังไม่มี
    cur.execute("INSERT INTO sectors (sector_name) VALUES (%s) ON CONFLICT (sector_name) DO NOTHING;", (sector_name,))
    
    # ดึง ID ของ Sector นั้นมา
    cur.execute("SELECT sector_id FROM sectors WHERE sector_name = %s", (sector_name,))
    s_id = cur.fetchone()[0]
    
    # 3.2 อัปเดตหุ้นแต่ละตัวให้ไปอยู่ Sector นั้น
    for sym in symbols:
        cur.execute("UPDATE stocks SET sector_id = %s WHERE symbol = %s;", (s_id, sym))

print("✅ อัปเดตข้อมูลเสร็จสิ้น! ตอนนี้หุ้นแยกกลุ่มกันชัดเจนแล้ว")

🔄 กำลังอัปเดต Sector ให้ถูกต้องตามความจริง...
✅ อัปเดตข้อมูลเสร็จสิ้น! ตอนนี้หุ้นแยกกลุ่มกันชัดเจนแล้ว


In [33]:
import psycopg2
import pandas as pd
from IPython.display import display, Markdown

# 1. เชื่อมต่อ Database
conn = psycopg2.connect(
    host="127.0.0.1",
    database="thai_stocks_db",
    user="postgres",
    password="Shifa.326459"
)

# ฟังก์ชันช่วยแสดงหัวข้อให้สวยงาม
def print_header(text):
    display(Markdown(f"### 📌 {text}"))

# =========================================================
# ส่วนที่ 1: ตอบคำถามเกี่ยวกับจำนวน (How many?)
# =========================================================

# Q1: How many stocks?
sql_q1 = "SELECT COUNT(*) AS \"Total Stocks (จำนวนหุ้นทั้งหมด)\" FROM stocks;"
df_q1 = pd.read_sql(sql_q1, conn)

# Q3: How many stock categories?
sql_q3 = "SELECT COUNT(*) AS \"Total Categories (จำนวนหมวดหมู่)\" FROM sectors;"
df_q3 = pd.read_sql(sql_q3, conn)

print_header("1. สรุปจำนวน (Overview Quantities)")
# แสดงผลคู่กัน
display(pd.concat([df_q1, df_q3], axis=1))


# =========================================================
# ส่วนที่ 2: รายชื่อหมวดหมู่ (List of Categories)
# =========================================================

# Q4: List of categories?
sql_q4 = """
SELECT sector_id AS "ID", sector_name AS "Industry Sector (ชื่อกลุ่ม)" 
FROM sectors 
ORDER BY sector_name;
"""
df_q4 = pd.read_sql(sql_q4, conn)

print_header("2. รายชื่อกลุ่มอุตสาหกรรม (List of Categories)")
display(df_q4)


# =========================================================
# ส่วนที่ 3: เจาะลึกรายชื่อหุ้น (Stock Lists)
# =========================================================

# Q5: List stock in each categories? (ไฮไลท์เด็ด: รวมชื่อหุ้นไว้ในช่องเดียว)
# ใช้คำสั่ง STRING_AGG ของ SQL ช่วยรวมชื่อหุ้นคั่นด้วยคอมม่า
sql_q5 = """
SELECT 
    s.sector_name AS "Category",
    COUNT(st.symbol) AS "Count",
    STRING_AGG(st.symbol, ', ') AS "Stock List (รายชื่อหุ้นในกลุ่ม)"
FROM sectors s
JOIN stocks st ON s.sector_id = st.sector_id
GROUP BY s.sector_name
ORDER BY "Count" DESC;
"""

# ตั้งค่า Pandas ให้โชว์ข้อความยาวๆ ได้ (ไม่ตัดเป็น ...)
pd.set_option('display.max_colwidth', None)
df_q5 = pd.read_sql(sql_q5, conn)

print_header("3. รายชื่อหุ้นแยกตามกลุ่ม (Stocks in Each Category)")
display(df_q5)


# =========================================================
# ส่วนที่ 4: รายชื่อหุ้นทั้งหมด (List all stock name)
# =========================================================

# Q2: List all stock name?

C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_7528\3813298693.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q1 = pd.read_sql(sql_q1, conn)
C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_7528\3813298693.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q3 = pd.read_sql(sql_q3, conn)


### 📌 1. สรุปจำนวน (Overview Quantities)

,Total Stocks (จำนวนหุ้นทั้งหมด),Total Categories (จำนวนหมวดหมู่)
0,43,11


C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_7528\3813298693.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q4 = pd.read_sql(sql_q4, conn)


### 📌 2. รายชื่อกลุ่มอุตสาหกรรม (List of Categories)

,ID,Industry Sector (ชื่อกลุ่ม)
0,45,BANK
1,46,COMM
2,44,ENERG
3,50,ETRON
4,53,FIN
5,51,FOOD
6,49,HELTH
7,47,ICT
8,52,PROP
9,1,SET100


C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_7528\3813298693.py:69: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q5 = pd.read_sql(sql_q5, conn)


### 📌 3. รายชื่อหุ้นแยกตามกลุ่ม (Stocks in Each Category)

,Category,Count,Stock List (รายชื่อหุ้นในกลุ่
0,ENERG,11,"PTT, PTTEP, TOP, GPSC, GULF, EA, BGRIM, EGCO, RATCH, OR, PTTGC"
1,BANK,6,"KBANK, SCB, BBL, KTB, TTB, TISCO"
2,FOOD,5,"CPF, TU, MINT, OSP, CBG"
3,COMM,4,"CPALL, CRC, HMPRO, GLOBAL"
4,SET100,3,"IVL, SCC, SCGP"
5,PROP,3,"CPN, LH, WHA"
6,FIN,3,"MTC, SAWAD, KTC"
7,TRANS,3,"AOT, BTS, BEM"
8,ICT,2,"ADVANC, TRUE"
9,HELTH,2,"BDMS, BH"
